# GRUPO 43 — Entrega 3: Bolsa de palabras binaria + Random Forest

**Idea de esta iteración:** probar un enfoque distinto al de las entregas 1 y 2 (TF‑IDF + regresión logística):

- **Representación:** bolsa de palabras **binaria** (`CountVectorizer(binary=True)`): 1 si la palabra aparece en la reseña, 0 si no. Sin pesos TF‑IDF.
- **Modelo:** **Random Forest**, un ensamble de árboles de decisión (práctica P10). Es no lineal, a diferencia de la regresión logística.
- Conservamos el hallazgo de la entrega 2: una representación para el texto completo y otra para la **última oración**.

Seguimos dentro de las reglas: BoW y scikit-learn, sin redes neuronales.

In [1]:
import re, unicodedata
from pathlib import Path
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, f1_score, classification_report

RANDOM_STATE = 42
DATA_DIR = Path("../data")
Path("models").mkdir(exist_ok=True)

train_df = pd.read_csv(DATA_DIR / "train.csv")
eval_df = pd.read_csv(DATA_DIR / "eval.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
print(train_df.shape, eval_df.shape)

(12000, 3) (3000, 2)


## 1. Preprocesamiento

Usamos las mismas funciones de la entrega 2: minúsculas, quitar tildes y dejar solo letras. Cada reseña se convierte en dos "vistas": el texto completo y la última oración.

In [2]:
def quitar_tildes(t):
    t = unicodedata.normalize("NFKD", t)
    return "".join(c for c in t if not unicodedata.combining(c))

def limpiar_texto(t):
    t = quitar_tildes(str(t).lower())
    t = re.sub(r"[^a-zñ\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def separar_oraciones(t):
    return [p for p in re.split(r"[.!?]+", str(t)) if limpiar_texto(p)]

def construir_vistas(textos):
    textos = pd.Series(textos).astype(str)
    ors = textos.map(separar_oraciones)
    return pd.DataFrame({
        "completo": textos.map(limpiar_texto).values,
        "ultima": ors.map(lambda o: limpiar_texto(o[-1]) if o else "").values,
    })

construir_vistas(train_df["text"].head(2))

,completo,ultima
0,lo pedi un martes por la noche y me llego al d...,viene con su cable y el manual en la caja
1,se lo compre a un familiar despues de pensarla...,al final de cuentas la nitidez de la imagen se...


## 2. Partición train / validación

Es la misma partición estratificada 80/20 de las entregas anteriores, para que los resultados sean comparables.

In [3]:
X_train, X_val, y_train, y_val = train_test_split(
    train_df["text"], train_df["label"], test_size=0.2,
    stratify=train_df["label"], random_state=RANDOM_STATE)
print(len(X_train), len(X_val))

9600 2400


## 3. Pipeline: BoW binaria + Random Forest

- `max_features` limita el vocabulario a las palabras más frecuentes. Random Forest escala peor que un modelo lineal con miles de columnas dispersas; además, con palabras muy raras cada árbol tiende a memorizar.
- `n_estimators=300` árboles; cada uno se entrena con una muestra *bootstrap* y en cada división evalúa un subconjunto aleatorio de palabras. Eso reduce la varianza frente a un solo árbol, que en la entrega 1 obtuvo ≈ 0.59.

In [4]:
def crear_pipeline(clf, max_completo=3000, max_ultima=2000):
    return Pipeline([
        ("vistas", FunctionTransformer(construir_vistas)),
        ("bow", ColumnTransformer([
            ("completo", CountVectorizer(binary=True, min_df=2, max_features=max_completo), "completo"),
            ("ultima",   CountVectorizer(binary=True, min_df=2, max_features=max_ultima),   "ultima"),
        ])),
        ("clf", clf),
    ])

rf = crear_pipeline(RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE))
rf.fit(X_train, y_train)
pred = rf.predict(X_val)
print(f"Random Forest -> accuracy: {accuracy_score(y_val, pred):.4f} | F1 macro: {f1_score(y_val, pred, average='macro'):.4f}")
print(classification_report(y_val, pred, digits=4))

Random Forest -> accuracy: 0.7750 | F1 macro: 0.7800
              precision    recall  f1-score   support

    negativo     0.7203    0.6690    0.6937       843
     neutral     0.9024    0.9958    0.9468       715
    positivo     0.7053    0.6936    0.6994       842

    accuracy                         0.7750      2400
   macro avg     0.7760    0.7861    0.7800      2400
weighted avg     0.7693    0.7750    0.7711      2400



**Qué sale:** Random Forest supera el 0.7 con holgura y mejora mucho frente al árbol de decisión solo de la entrega 1 (≈ 0.59). Aun así queda por debajo de la entrega 2 (≈ 0.89). Lo esperable: cada división de un árbol mira **una sola palabra**, mientras que el sentimiento se arma sumando evidencia de muchas palabras a la vez, que es justo lo que hace bien un modelo lineal.

## 4. Pequeña prueba de hiperparámetros

Probamos pocas combinaciones de `max_depth` y `min_samples_leaf`, que controlan qué tanto crece cada árbol (complejidad y sobreajuste). Se evalúan sobre validación; es una exploración rápida, no una búsqueda exhaustiva.

In [5]:
filas = []
for depth in [None, 60]:
    for leaf in [1, 2]:
        m = crear_pipeline(RandomForestClassifier(n_estimators=300, max_depth=depth, min_samples_leaf=leaf,
                                                  n_jobs=-1, random_state=RANDOM_STATE)).fit(X_train, y_train)
        p = m.predict(X_val)
        filas.append({"max_depth": depth, "min_samples_leaf": leaf,
                      "acc_train": accuracy_score(y_train, m.predict(X_train)),
                      "f1_val": f1_score(y_val, p, average="macro")})
res = pd.DataFrame(filas).round(4).sort_values("f1_val", ascending=False)
res

,max_depth,min_samples_leaf,acc_train,f1_val
1,NaN,2,0.9950,0.7921
3,60.0,2,0.9935,0.7826
0,NaN,1,1.0000,0.7800
2,60.0,1,1.0000,0.7783


**Qué sale:** con árboles sin límite la exactitud en entrenamiento es casi 1, porque cada árbol memoriza. El ensamble compensa parte de ese sobreajuste al promediar. Las diferencias en validación entre configuraciones son pequeñas.

## 5. Referencia rápida: misma BoW binaria con Naive Bayes Bernoulli

Para saber si el límite está en la representación o en el modelo, usamos la **misma** bolsa de palabras binaria con `BernoulliNB`, el Naive Bayes pensado para variables binarias.

In [6]:
nb = crear_pipeline(BernoulliNB(), max_completo=None, max_ultima=None).fit(X_train, y_train)
p_nb = nb.predict(X_val)
print(f"BernoulliNB -> accuracy: {accuracy_score(y_val, p_nb):.4f} | F1 macro: {f1_score(y_val, p_nb, average='macro'):.4f}")

BernoulliNB -> accuracy: 0.8533 | F1 macro: 0.8598


**Qué sale:** con la misma representación, Naive Bayes rinde más que Random Forest. Entonces la BoW binaria sí contiene la información necesaria, y lo que limita es el modelo de árboles en este tipo de datos dispersos.

## 6. Modelo final de esta entrega y submission

El objetivo de esta iteración es probar el enfoque de **ensamble de árboles**, así que el envío es el de Random Forest con la mejor configuración de la tabla anterior, reentrenado con todo `train.csv`.

In [7]:
mejor = res.iloc[0]
depth = None if pd.isna(mejor["max_depth"]) else int(mejor["max_depth"])
modelo_final = crear_pipeline(RandomForestClassifier(
    n_estimators=300, max_depth=depth, min_samples_leaf=int(mejor["min_samples_leaf"]),
    n_jobs=-1, random_state=RANDOM_STATE))
modelo_final.fit(train_df["text"], train_df["label"])

submission = pd.DataFrame({"id": eval_df["id"], "answer": modelo_final.predict(eval_df["text"])})
assert list(submission.columns) == list(sample_sub.columns) and len(submission) == len(sample_sub)
submission.to_csv("submission-G43-3.csv", index=False)
joblib.dump(modelo_final, "models/modelo_entrega3_bow_randomforest.joblib")
print(submission["answer"].value_counts(normalize=True).round(3))

answer
positivo    0.348
negativo    0.340
neutral     0.312
Name: proportion, dtype: float64


## Resumen de la iteración

| Entrega | Representación | Modelo | F1 macro validación |
|---|---|---|---|
| 1 | TF‑IDF texto completo | Regresión logística | ≈ 0.75 |
| 2 | TF‑IDF texto completo + última oración | Regresión logística | ≈ 0.89 |
| 3 | BoW binaria texto completo + última oración | Random Forest | ≈ 0.79 |
| 3 (referencia) | BoW binaria texto completo + última oración | Bernoulli Naive Bayes | ≈ 0.86 |

**Aprendizajes:** (1) la idea de la última oración funciona con cualquier representación; (2) en texto disperso con muchas variables, los modelos lineales o probabilísticos (regresión logística, Naive Bayes) rinden mejor que los ensambles de árboles; (3) para la entrega final vale la pena considerar un ensamble por votación de los mejores modelos.